# 01 · Load and check the data

**Project:** Early detection of at-risk students through performance-trajectory modeling
**Goal of this notebook:** load the raw UCI dataset, confirm it is the file we expect, and record its basic structure. There is no analysis here yet; exploratory analysis is in `02_eda.ipynb`.

In [1]:
import sys
from pathlib import Path

# Make `src/` importable when the notebook runs from the notebooks/ folder.
sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd
from dropout import config
from dropout.data import file_sha256, load_data

pd.set_option("display.max_columns", 50)

## 1. Load and validate
`load_data()` reads the `;`-separated file, cleans the column names, and raises an error if the shape, columns, missing values or target classes are not what we expect.

In [2]:
df = load_data()
print("Shape:", df.shape)
print("Matches expected (4424, 37):", df.shape == (config.EXPECTED_N_ROWS, config.EXPECTED_N_COLS))
print("Raw file checksum matches:", file_sha256(config.RAW_DATA_PATH) == config.RAW_DATA_SHA256)

Shape: (4424, 37)
Matches expected (4424, 37): True
Raw file checksum matches: True


37 columns = **36 features + 1 target**.

In [3]:
df.head()

,marital_status,application_mode,application_order,course,daytime_attendance,previous_qualification,previous_qualification_grade,nationality,mother_qualification,father_qualification,mother_occupation,father_occupation,admission_grade,displaced,special_needs,debtor,tuition_up_to_date,gender,scholarship_holder,age_at_enrollment,international,sem1_credited,sem1_enrolled,sem1_evaluations,sem1_approved,sem1_grade,sem1_without_evaluations,sem2_credited,sem2_enrolled,sem2_evaluations,sem2_approved,sem2_grade,sem2_without_evaluations,unemployment_rate,inflation_rate,gdp,target
0,1,17,5,171,1,1,122.0,1,19,12,5,9,127.3,1,0,0,1,1,0,20,0,0,0,0,0,0.000000,0,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,3,3,142.5,1,0,0,0,1,0,19,0,0,6,6,6,14.000000,0,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,9,9,124.8,1,0,0,0,1,0,19,0,0,6,0,0,0.000000,0,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,5,3,119.6,1,0,0,1,0,0,20,0,0,6,8,6,13.428571,0,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,9,9,141.5,0,0,0,1,0,0,45,0,0,6,9,5,12.333333,0,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


## 2. Column types
Every column is numeric except `target`. Several integer columns are actually category codes (see `config.CATEGORICAL_FEATURES`).

In [4]:
print(df.dtypes.value_counts().to_string())
print()
print("Categorical (coded) features:", len(config.CATEGORICAL_FEATURES))

int64      29
float64     7
str         1

Categorical (coded) features: 9


## 3. Missing values and duplicates

In [5]:
print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))

Missing values: 0
Duplicate rows: 0


## 4. Target distribution

In [6]:
counts = df[config.TARGET].value_counts()
pd.DataFrame({"students": counts, "share": (counts / len(df)).round(3)})

,students,share
target,,
Graduate,2209,0.499
Dropout,1421,0.321
Enrolled,794,0.179


## 5. Features by stage
The features are grouped by when a school would have them. These groups are the three modeling stages.

In [7]:
pd.DataFrame(
    {"n_features": {stage: len(cols) for stage, cols in config.STAGES.items()}}
)

,n_features
enrollment,24
semester_1,30
semester_2,36


## Summary

- The data loads correctly: **4,424 rows × 37 columns** (36 features + target), and the file's checksum matches.
- There are no missing values and no duplicate rows.
- Classes: Graduate 2,209 (49.9%), Dropout 1,421 (32.1%), Enrolled 794 (17.9%).
- Stage groups: 24 features at enrollment, +6 after semester 1, +6 after semester 2.